## Grupo 1 | MCDI500 - Programación para la ciencia de datos

### Integrantes

- Pablo Ignacio Balbontin Constenla @pabbalbontin-maker
- Melany Esmeralda Reyes Leiva @melanyreyesy
- Ingeborg Andrea Munoz Carnot @dark452
- Mario Alejandro Lopez Pulgar @malp2203

### Descripción del proyecto - Fase 2
El objetivo del proyecto es realizar un análisis del impacto la utilización y frecuencia de uso de la IA generativa sobre el rendimiento académico y el nivel de agotamiento de
estudiantes universitarios. En esta fase nos concentraremos en explorar el dataset en profundidad, limpiar y transformar los datos para dejarlos listos para la etapa de modelado.

### Objetivo General
Construir una pipeline de datos reproducible que realice las siguientes operaciones sobre el dataset:

- Limpieza
- Codificado
- Escalado
- Preparado para modelado posterior

### Objetivos Específicos

- Obtener y explorar los datos verificando su estructura y calidad.
- Limpiar gestionando rigurosamente los valores nulos.
- Transformar las variables categóricas nominales con codificación One-Hot.
- Estandarizar las variables continuas.
- Validar técnicamente el resultado.

**Atributos:** `Student_ID`, `Major_Category`, `Year_of_Study`, `Pre_Semester_GPA`, `Weekly_GenAI_Hours`, `Primary_Use_Case`,
`Prompt_Engineering_Skill`, `Tool_Diversity`, `Paid_Subscription`, `Traditional_Study_Hours`, `Perceived_AI_Dependency`, `Institutional_Policy`, `Anxiety_Level_During_Exams`, `Skill_Retention_Score`, además las variables
objetivo `Burnout_Risk_Level`(High, Low y Medium) y `Post_Semester_GPA`.

### Librerías utilizadas

| Librería | Función |
|---|---|
| `pandas` | Cargar y manipular la tabla de datos (el `DataFrame`). |
| `sklearn.preprocessing` | Las herramientas de codificación (`LabelEncoder`, `OneHotEncoder`) y escalamiento (`StandardScaler`). |

`np.random.seed(42)` fija la semilla aleatoria: garantiza que cualquier proceso con azar dé **siempre el mismo resultado**, lo que asegura la *reproducibilidad* exigida en la fase.

### Importar librerías
Se importan las librería necesarias para cargar y manipular el dataset

In [ ]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler, StandardScaler

### Función Carga de dataset
Se crea la función que carga el dataset utilizando la función DataFrame de pandas (es una estructura de datos bidimensional y tabular).

In [ ]:
def load_data(file_path: str) -> pd.DataFrame:
    """Carga dataset raw, desde un archivo CSV

    Parametros
    ----------
    file_path : str
        Ruta del archivo CSV utilizado como entrada.

    Retorno
    -------
    pd.DataFrame
        Datos cargados en un DataFrame.

    Excepción
    ---------
    FileNotFoundError
        Si la ruta al archivp CSV no existe. Se muestra un mensaje de error
    """
    try:
        df = pd.read_csv(file_path)
    except FileNotfoundError:
        raise FileNotFoundError(
            f"No se encontró el archivo '{file_path}'."
            "Verificar que el archivo CSV se encuentre en data/raw."
        )
    df = pd.read_csv(file_path)
    return df

df = load_data('../data/raw/ai_student_impact_dataset.csv')

## Visualización de datos
### Función Mostrar dimensiones
Se crea la función para mostrar dimensiones y tipos de datos del dataset

La función `show_tipos` muestra la siguiente información:

1. **Dimensiones** (`shape`): Cantidad de columnas y filas
2. **Tipos de datos** (`dtypes`): Qué columnas son numéricas y cuáles son texto. Las de texto (object) son las que tendremos que codificar más adelante.


In [ ]:
def show_tipos(df: pd.DataFrame) -> None:
    """Muestra dimensiones y tipos de datos del dataset."""
    print(f'Dimensiones: {df.shape}')
    print()
    print(f"{'Columna':<27} {'Tipo Dato'}")
    print("-" * 37)
    print(df.dtypes)
    print()

show_tipos(df)


### Función + visualizacion de valores Nulos
Se crea la función que muestra los valores nulos por columna

1. Valores nulos ( `isnull().sum())`: Cuántos datos faltan en cada columna. Aquí se detecta que no hay columnas con valores nulos.

In [ ]:
def show_nulos(df: pd.DataFrame) -> None:
    """Muestra valores nulos por columna."""
    print('Valores nulos por columna:')
    print("-" * 31)
    print(df.isnull().sum())
    print()

show_nulos(df)

### Función + visualizacion de Estadísticas Descriptivas

1.- Estadísticas descriptivas (`describe()`): Media, mínimo, máximo, etc., de las variables numéricas, útil para detectar rangos raros o valores atípicos.

Se muestra un resumen de las columnas con las siguientes metricas:
- count: cantidad de valores no nulos
- mean: promedio 
- std: desviación estándar
- min: valor mínimo
- 25%: primer cuartil
- 50%: mediana
- 75%: tercer cuartil
- max:valor máximo

In [ ]:
def show_estadisticas(df: pd.DataFrame) -> None:
    """Muestra estadísticas descriptivas."""
    print('Estadísticas descriptivas:')
    print("-" * 31)
    print(df.describe())

show_estadisticas(df)

### Función + visualizacion de frecuencia de Categorías por columna
Conteo de categorias en las variables nominales (control de calidad de entrada)

In [ ]:
def show_categories(df: pd.DataFrame) -> None:
  """Muestra la repetición de las variables por columna."""
  cat_cols = df.select_dtypes(include="object").columns.tolist()
  for col in cat_cols:
    print(f"\n{col}:")
    print(df[col].value_counts().to_string())

show_categories(df)

### Limpieza de datos

Limpiar los datos significa dejar los datos completos y consistentes. Tomamos dos decisiones, cada
una justificada técnicamente:

**1) Eliminar `Student_ID`.** Es un identificador único (un número distinto por estudiante). No
contiene información que ayude a predecir el rendimiento académico ni el nivel de agotamiento de los estudiantes universitarios, así que lo
descartamos para que no introduzca ruido.

El dataset no contiene valores nulos en ninguna columna, por lo tanto no es necesario imputar una columna en particular.

Eliminamos el identificador `Student_ID`

In [ ]:
df = df.drop(columns=['Student_ID'])
df.head()

### Conversión de variable booleana

`Paid_Subscription` es de tipo `bool`. La conversión explícita a `int64` garantiza compatibilidad con todos los estimadores de scikit-learn.

In [ ]:
def cast_bool_to_int(df: pd.DataFrame, col: str = 'Paid_Subscription') -> pd.DataFrame:
    """Convierte columna booleana a entero (False->0, True->1).

    Parametros
    ----------
    df  : pd.DataFrame
    col : nombre de la columna booleana (default 'Paid_Subscription')

    Retorna
    -------
    pd.DataFrame con la columna convertida a int64
    """
    if col not in df.columns:
        print(f'[WARN] Columna {col} no encontrada.')
        return df
    print(f'[INFO] {col} — dtype antes: {df[col].dtype}')
    # Conversion via numpy array, evita inferencia de tipos de pandas
    df[col] = df[col].to_numpy().astype(int)
    print(f'[INFO] {col} — dtype despues: {df[col].dtype}')
    if df[col].dtype != 'int64':
        raise TypeError(
            f'La conversion no produjo int64. dtype resultante: {df[col].dtype}.'
        )
    conteo = df[col].value_counts().sort_index().to_dict()
    print(f'[OK] {col} convertida a int64. Distribucion: {conteo}')
    return df

df = cast_bool_to_int(df)
df.head()

## 5. Codificación de variables ordinales

Las variables ordinales poseen una relación de orden inherente que los modelos pueden aprovechar. Se asignan enteros consecutivos según jerarquía real.

Aplicar One-Hot Encoding a variables ordinales destruiría la relación de orden y expandiría innecesariamente el espacio de features.

| Variable | Tipo | Mapping |
|---|---|---|
| `Year_of_Study` | Ordinal | Freshman=0, Sophomore=1, Junior=2, Senior=3, Graduate=4 |
| `Prompt_Engineering_Skill` | Ordinal | Beginner=0, Intermediate=1, Advanced=2 |
| `Burnout_Risk_Level` | Ordinal (target clasificación) | Low=0, Medium=1, High=2 |

In [ ]:
def encode_ordinal(df: pd.DataFrame, col: str, order: list) -> pd.DataFrame:
    """Codifica variable ordinal asignando enteros segun el orden definido.

    Parametros
    ----------
    df    : pd.DataFrame
    col   : nombre de la columna a codificar
    order : lista de categorias en orden ascendente
            Ejemplo: ['Low', 'Medium', 'High'] -> {Low:0, Medium:1, High:2}

    Retorna
    -------
    pd.DataFrame con la columna reemplazada por valores enteros
    """
    if col not in df.columns:
        print(f'[WARN] Columna {col} no encontrada.')
        return df
    mapping = {cat: idx for idx, cat in enumerate(order)}
    df[col] = df[col].map(mapping)
    nulos_post = df[col].isnull().sum()
    if nulos_post > 0:
        print(f'[ERROR] {col}: {nulos_post} valores sin mapeo. Verificar order.')
    else:
        print(f'[OK] {col} codificada. Mapping: {mapping}')
    return df

In [ ]:
df = encode_ordinal(df, 'Year_of_Study',
                    order=['Freshman', 'Sophomore', 'Junior', 'Senior', 'Graduate'])

df = encode_ordinal(df, 'Prompt_Engineering_Skill',
                    order=['Beginner', 'Intermediate', 'Advanced'])

df = encode_ordinal(df, 'Burnout_Risk_Level',
                    order=['Low', 'Medium', 'High'])
df.head()